# **🔍 Clean & Jerk Dataset — Per-Annotator Inter-Annotator SD Diagnostics**

Self-contained diagnostic notebook comparing **each individual annotator's contributed SD distribution** against the **overall pooled SD distribution** (`n_annotators >= 3`), plus **Leave-One-Out (LOO) SD impact** to identify whether any specific annotator systematically inflates inter-annotator SD.

**Pipeline:** `sets/set_*.h5 (phase_markers/annotators + consensus) → Join on (set_id, lift_id, marker_name) → Individual vs. Overall SD Distributions & LOO ΔSD`

- **Contributed SD Distribution:** Normalized histogram (`histnorm="probability"`) of `sd_ms` for all marker instances annotated by annotator $a$, plotted directly against the overall pooled distribution.
- **Leave-One-Out Impact ($\Delta\text{SD}_a = \text{SD}_{\text{all}} - \text{SD}_{-a}$):** Positive values mean removing annotator $a$ *reduces* SD (i.e. $a$ drove SD higher); negative values mean $a$ agreed closely with consensus.
- **Signed Bias vs. Peers ($t_a - \bar{t}_{-a}$):** Detects systematic early (negative) or late (positive) frame placement relative to the other annotators on the same lift/marker.

### 🔧 Colab Setup & Library Init

In [1]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, importlib.util, os, subprocess, sys, types

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

dataset_dir = "/content/drive/MyDrive/Academic/Research/Wearable Sensing/Wearable Biomechanics/Clean&JerkDataset" # @param {type:"string"}
EXPORT_CFG = {"toImageButtonOptions": {"format": "png", "scale": 4}}

### 🧮 Self-Contained Loader, LOO Metrics & Plotting Functions

In [2]:
# @title 🧮 Diagnostic Loader & Plotting Functions { display-mode: "form" }
from __future__ import annotations

import math
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_FONT = dict(family="Arial, Helvetica, sans-serif", size=12, color="#1f2937")
_PALETTE = [
    "#3B82F6", "#E08B63", "#52B79A", "#8B5CF6", "#F59E0B",
    "#EC4899", "#06B6D4", "#84CC16", "#EF4444", "#6366F1",
    "#14B8A6", "#F97316", "#A855F7", "#0EA5E9", "#64748B",
]


def _decode_arr(arr) -> list[str]:
    return [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in arr]


def _annotator_sort_key(aid: str) -> tuple[int, str]:
    m = re.search(r"(\d+)", str(aid))
    return (int(m.group(1)) if m else 9999, str(aid))


def _style_axes(fig: go.Figure) -> None:
    fig.update_xaxes(
        showline=True, linewidth=1, linecolor="#cbd5e1",
        showgrid=True, gridcolor="#eef2f7", zeroline=False,
    )
    fig.update_yaxes(
        showline=True, linewidth=1, linecolor="#cbd5e1",
        showgrid=True, gridcolor="#eef2f7", zeroline=False,
    )


def load_annotator_sd_data(
    dataset_dir: str | Path,
    min_annotators: int = 3,
    default_fps: float = 30.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Load per-annotator marker records joined with consensus SD and LOO ΔSD across all sets.

    Returns
    -------
    ann_sd_df : pd.DataFrame
        One row per (set_id, lift_id, marker_name, annotator_id) for instances with >= min_annotators.
        Includes `sd_ms`, `loo_sd_ms`, `loo_delta_sd_ms` (= sd_all - sd_without_a), and `bias_vs_peers_ms`.
    consensus_all_df : pd.DataFrame
        One row per (set_id, lift_id, marker_name) with >= min_annotators (the overall reference pool).
    '''
    sets_dir = Path(dataset_dir) / "sets"
    h5_files = sorted(sets_dir.glob("set_*.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No set_*.h5 files found under: {sets_dir}")

    cons_parts: list[pd.DataFrame] = []
    ann_parts: list[pd.DataFrame] = []

    for h5_path in h5_files:
        set_id = h5_path.stem
        with h5py.File(h5_path, "r") as f:
            if "phase_markers/consensus" not in f or "phase_markers/annotators" not in f:
                continue
            fps = float(f["metadata"].attrs.get("video_fps", default_fps)) if "metadata" in f else default_fps
            if not np.isfinite(fps) or fps <= 0:
                fps = default_fps
            ms_per_frame = 1000.0 / fps

            cons = pd.DataFrame(
                {
                    k: (_decode_arr(v[:]) if v.dtype.kind in {"S", "O"} else v[:])
                    for k, v in f["phase_markers/consensus"].items()
                }
            )
            ann = pd.DataFrame(
                {
                    k: (_decode_arr(v[:]) if v.dtype.kind in {"S", "O"} else v[:])
                    for k, v in f["phase_markers/annotators"].items()
                }
            )

        if cons.empty or ann.empty:
            continue

        cons["set_id"] = set_id
        ann["set_id"] = set_id
        ann["ms_per_frame"] = ms_per_frame
        cons_parts.append(cons)
        ann_parts.append(ann)

    if not cons_parts:
        raise RuntimeError("No phase_markers data found in any .h5 file.")

    consensus_all_df = pd.concat(cons_parts, ignore_index=True)
    consensus_all_df = consensus_all_df[consensus_all_df["n_annotators"] >= min_annotators].dropna(subset=["sd_ms"]).copy()

    ann_df = pd.concat(ann_parts, ignore_index=True)
    # Deduplicate if an annotator has repeated rows on the same key
    ann_df = ann_df.drop_duplicates(subset=["set_id", "lift_id", "marker_name", "annotator_id"])

    merged = ann_df.merge(
        consensus_all_df[["set_id", "lift_id", "marker_name", "n_annotators", "sd_ms", "sd_frames", "mean_frame"]],
        on=["set_id", "lift_id", "marker_name"],
        how="inner",
    )

    # Compute Leave-One-Out SD impact and signed bias vs. peers per (set_id, lift_id, marker_name)
    loo_sd_list = np.full(len(merged), np.nan, dtype=float)
    loo_delta_list = np.full(len(merged), np.nan, dtype=float)
    bias_peers_list = np.full(len(merged), np.nan, dtype=float)

    grouped = merged.groupby(["set_id", "lift_id", "marker_name"], sort=False)
    for _, idx in grouped.groups.items():
        pos = merged.index.get_indexer(idx)
        frames = merged.iloc[pos]["frame"].to_numpy(dtype=float)
        mpf = float(merged.iloc[pos[0]]["ms_per_frame"])
        k = len(frames)
        if k < 3:
            continue
        sd_all_calc = float(np.std(frames, ddof=1) * mpf)
        for i in range(k):
            others = np.delete(frames, i)
            sd_wo = float(np.std(others, ddof=1) * mpf)
            loo_sd_list[pos[i]] = sd_wo
            loo_delta_list[pos[i]] = sd_all_calc - sd_wo
            bias_peers_list[pos[i]] = (frames[i] - float(np.mean(others))) * mpf

    merged["loo_sd_ms"] = loo_sd_list
    merged["loo_delta_sd_ms"] = loo_delta_list
    merged["bias_vs_peers_ms"] = bias_peers_list
    return merged, consensus_all_df


def build_annotator_summary_table(
    ann_sd_df: pd.DataFrame,
    consensus_all_df: pd.DataFrame,
    threshold_ms: float = 200.0,
) -> pd.DataFrame:
    '''Summary table comparing each annotator's contributed SD distribution against overall.'''
    sd_overall = consensus_all_df["sd_ms"].dropna().to_numpy(dtype=float)
    ov_med = float(np.median(sd_overall))
    ov_q25, ov_q75, ov_p95 = np.percentile(sd_overall, [25, 75, 95])

    rows = []
    for aid in sorted(ann_sd_df["annotator_id"].unique(), key=_annotator_sort_key):
        sub = ann_sd_df[ann_sd_df["annotator_id"] == aid]
        vals = sub["sd_ms"].dropna().to_numpy(dtype=float)
        loo_d = sub["loo_delta_sd_ms"].dropna().to_numpy(dtype=float)
        bias = sub["bias_vs_peers_ms"].dropna().to_numpy(dtype=float)
        q25, med, q75, p95 = np.percentile(vals, [25, 50, 75, 95])
        rows.append(
            {
                "Annotator": aid,
                "Sets": int(sub["set_id"].nunique()),
                "Instances": int(len(vals)),
                "Median SD (ms)": round(float(med), 1),
                "Δ Med vs Overall (ms)": round(float(med - ov_med), 1),
                "IQR (ms)": f"[{q25:.1f}, {q75:.1f}]",
                "95th pct (ms)": round(float(p95), 1),
                f"% > {int(threshold_ms)} ms": round(float((vals > threshold_ms).mean() * 100.0), 1),
                "Mean LOO ΔSD (ms)": round(float(np.mean(loo_d)), 2) if len(loo_d) else np.nan,
                "Median LOO ΔSD (ms)": round(float(np.median(loo_d)), 2) if len(loo_d) else np.nan,
                "Mean Bias vs Peers (ms)": round(float(np.mean(bias)), 1) if len(bias) else np.nan,
            }
        )

    df = pd.DataFrame(rows).sort_values("Median SD (ms)", ascending=False).reset_index(drop=True)
    overall_row = pd.DataFrame(
        [
            {
                "Annotator": "OVERALL (All)",
                "Sets": int(consensus_all_df["set_id"].nunique()),
                "Instances": int(len(sd_overall)),
                "Median SD (ms)": round(ov_med, 1),
                "Δ Med vs Overall (ms)": 0.0,
                "IQR (ms)": f"[{ov_q25:.1f}, {ov_q75:.1f}]",
                "95th pct (ms)": round(float(ov_p95), 1),
                f"% > {int(threshold_ms)} ms": round(float((sd_overall > threshold_ms).mean() * 100.0), 1),
                "Mean LOO ΔSD (ms)": 0.0,
                "Median LOO ΔSD (ms)": 0.0,
                "Mean Bias vs Peers (ms)": 0.0,
            }
        ]
    )
    return pd.concat([overall_row, df], ignore_index=True)


def plot_annotator_sd_small_multiples(
    ann_sd_df: pd.DataFrame,
    consensus_all_df: pd.DataFrame,
    bin_width_ms: float = 20.0,
    sort_by: str = "median_sd",
    x_max_ms: float | None = None,
    cols: int = 3,
) -> go.Figure:
    '''Small-multiples grid: each annotator's contributed SD distribution overlaid on overall.'''
    sd_all = consensus_all_df["sd_ms"].dropna().to_numpy(dtype=float)
    ov_med = float(np.median(sd_all))
    if x_max_ms is None:
        x_max_ms = float(np.ceil(np.percentile(sd_all, 99.5) / bin_width_ms) * bin_width_ms)

    # Determine annotator ordering
    aids = list(ann_sd_df["annotator_id"].unique())
    if sort_by == "median_sd":
        aids.sort(key=lambda a: float(ann_sd_df.loc[ann_sd_df["annotator_id"] == a, "sd_ms"].median()), reverse=True)
    elif sort_by == "mean_loo_delta_sd":
        aids.sort(key=lambda a: float(ann_sd_df.loc[ann_sd_df["annotator_id"] == a, "loo_delta_sd_ms"].mean()), reverse=True)
    else:
        aids.sort(key=_annotator_sort_key)

    n_ann = len(aids)
    rows = math.ceil(n_ann / cols)

    titles = []
    for aid in aids:
        sub = ann_sd_df[ann_sd_df["annotator_id"] == aid]
        med = float(sub["sd_ms"].median())
        d_med = med - ov_med
        loo_m = float(sub["loo_delta_sd_ms"].mean())
        sign = "+" if d_med >= 0 else ""
        loo_sign = "+" if loo_m >= 0 else ""
        titles.append(
            f"<b>{aid}</b> (n={len(sub)} | Med {med:.1f} ms [{sign}{d_med:.1f}] | LOO {loo_sign}{loo_m:.1f})"
        )

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=titles,
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.05,
        vertical_spacing=max(0.06, min(0.14, 0.55 / max(rows, 1))),
    )

    xbins = dict(start=0.0, end=float(x_max_ms), size=float(bin_width_ms))

    for idx, aid in enumerate(aids):
        r = idx // cols + 1
        c = idx % cols + 1
        sub_vals = ann_sd_df.loc[ann_sd_df["annotator_id"] == aid, "sd_ms"].dropna().to_numpy(dtype=float)
        med = float(np.median(sub_vals))
        d_med = med - ov_med
        color = "#E74C3C" if d_med > 15.0 else ("#F59E0B" if d_med > 5.0 else "#3B82F6")

        # Overall reference histogram (muted gray backdrop)
        fig.add_trace(
            go.Histogram(
                x=sd_all,
                xbins=xbins,
                histnorm="probability",
                marker_color="rgba(148, 163, 184, 0.42)",
                marker_line=dict(width=0.6, color="#64748b"),
                name="Overall (All)",
                legendgroup="overall",
                showlegend=(idx == 0),
                hovertemplate="Overall<br>SD: %{x:.1f} ms<br>Prop: %{y:.3f}<extra></extra>",
            ),
            row=r,
            col=c,
        )
        # Individual annotator histogram
        fig.add_trace(
            go.Histogram(
                x=sub_vals,
                xbins=xbins,
                histnorm="probability",
                marker_color=color,
                opacity=0.62,
                marker_line=dict(width=0.7, color="#1f2937"),
                name="Annotator (contributed)",
                legendgroup="ann",
                showlegend=(idx == 0),
                hovertemplate=f"{aid}<br>SD: %{{x:.1f}} ms<br>Prop: %{{y:.3f}}<extra></extra>",
            ),
            row=r,
            col=c,
        )

        # Reference vertical lines: Overall median (gray dashed) & Annotator median (solid colored)
        fig.add_vline(x=ov_med, line_width=1.5, line_dash="dash", line_color="#475569", row=r, col=c)
        fig.add_vline(x=med, line_width=2.0, line_dash="solid", line_color=color, row=r, col=c)

    fig.update_layout(
        barmode="overlay",
        title=dict(
            text=(
                f"<b>Per-Annotator Contributed SD Distribution vs. Overall</b>"
                f"<br><sup>Gray = Overall (Median {ov_med:.1f} ms, dashed) | "
                f"Blue = near/below overall, Amber = +5..15 ms, Red = >+15 ms above overall median</sup>"
            ),
            x=0.5,
            font=dict(size=14),
        ),
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=_FONT,
        height=max(360, 260 * rows + 110),
        margin=dict(l=60, r=25, t=95, b=55),
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1.0),
    )
    _style_axes(fig)
    fig.update_xaxes(range=[0, x_max_ms])
    for c in range(1, cols + 1):
        fig.update_xaxes(title_text="Inter-Annotator SD (ms)", row=rows, col=c)
    for r in range(1, rows + 1):
        fig.update_yaxes(title_text="Proportion", row=r, col=1)
    fig.update_annotations(font_size=10)
    return fig


def plot_annotator_sd_overlay(
    ann_sd_df: pd.DataFrame,
    consensus_all_df: pd.DataFrame,
    bin_width_ms: float = 20.0,
    x_max_ms: float | None = None,
) -> go.Figure:
    '''Two-panel interactive overlay: (1) Normalized SD Density Curves, (2) Empirical CDFs.'''
    sd_all = consensus_all_df["sd_ms"].dropna().to_numpy(dtype=float)
    if x_max_ms is None:
        x_max_ms = float(np.ceil(np.percentile(sd_all, 99.5) / bin_width_ms) * bin_width_ms)

    edges = np.arange(0.0, x_max_ms + bin_width_ms, bin_width_ms)
    centers = 0.5 * (edges[:-1] + edges[1:])

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "<b>Normalized SD Distribution (Click Legend to Isolate)</b>",
            "<b>Empirical CDF (Lower Curve = Higher SD)</b>",
        ),
        horizontal_spacing=0.08,
    )

    # Overall reference curve
    ov_hist, _ = np.histogram(sd_all, bins=edges)
    ov_prop = ov_hist / max(len(sd_all), 1)
    ov_sorted = np.sort(sd_all)
    ov_cdf = np.arange(1, len(ov_sorted) + 1) / len(ov_sorted)

    fig.add_trace(
        go.Scatter(
            x=centers,
            y=ov_prop,
            mode="lines",
            name="OVERALL (All)",
            legendgroup="OVERALL",
            line=dict(color="#111827", width=3.2),
            hovertemplate="OVERALL<br>SD: %{x:.1f} ms<br>Prop: %{y:.3f}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=ov_sorted,
            y=ov_cdf,
            mode="lines",
            name="OVERALL (All)",
            legendgroup="OVERALL",
            showlegend=False,
            line=dict(color="#111827", width=3.2),
            hovertemplate="OVERALL<br>SD: %{x:.1f} ms<br>CDF: %{y:.3f}<extra></extra>",
        ),
        row=1,
        col=2,
    )

    aids = sorted(ann_sd_df["annotator_id"].unique(), key=_annotator_sort_key)
    for idx, aid in enumerate(aids):
        vals = ann_sd_df.loc[ann_sd_df["annotator_id"] == aid, "sd_ms"].dropna().to_numpy(dtype=float)
        color = _PALETTE[idx % len(_PALETTE)]
        h, _ = np.histogram(vals, bins=edges)
        prop = h / max(len(vals), 1)
        s_vals = np.sort(vals)
        cdf = np.arange(1, len(s_vals) + 1) / len(s_vals)

        fig.add_trace(
            go.Scatter(
                x=centers,
                y=prop,
                mode="lines",
                name=f"{aid} (n={len(vals)})",
                legendgroup=aid,
                line=dict(color=color, width=1.6),
                opacity=0.82,
                hovertemplate=f"{aid}<br>SD: %{{x:.1f}} ms<br>Prop: %{{y:.3f}}<extra></extra>",
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=s_vals,
                y=cdf,
                mode="lines",
                name=f"{aid} (n={len(vals)})",
                legendgroup=aid,
                showlegend=False,
                line=dict(color=color, width=1.6),
                opacity=0.82,
                hovertemplate=f"{aid}<br>SD: %{{x:.1f}} ms<br>CDF: %{{y:.3f}}<extra></extra>",
            ),
            row=1,
            col=2,
        )

    fig.update_layout(
        title=dict(text="<b>Interactive Overlay: Individual Annotators vs. Overall SD Distribution</b>", x=0.5, font=dict(size=14)),
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=_FONT,
        height=470,
        margin=dict(l=60, r=25, t=75, b=55),
    )
    _style_axes(fig)
    fig.update_xaxes(title_text="Inter-Annotator SD (ms)", range=[0, x_max_ms], row=1, col=1)
    fig.update_xaxes(title_text="Inter-Annotator SD (ms)", range=[0, x_max_ms], row=1, col=2)
    fig.update_yaxes(title_text="Proportion of Instances", row=1, col=1)
    fig.update_yaxes(title_text="Cumulative Proportion", range=[0, 1.02], row=1, col=2)
    return fig


def plot_annotator_loo_impact(ann_sd_df: pd.DataFrame) -> go.Figure:
    '''Two-panel causal diagnostic: (1) Mean LOO ΔSD per annotator, (2) Signed Bias vs. Peers.'''
    agg = (
        ann_sd_df.groupby("annotator_id", as_index=False)
        .agg(
            n=("sd_ms", "count"),
            mean_loo=("loo_delta_sd_ms", "mean"),
            med_loo=("loo_delta_sd_ms", "median"),
            se_loo=("loo_delta_sd_ms", lambda s: float(np.std(s.dropna(), ddof=1) / np.sqrt(max(len(s.dropna()), 1)))),
        )
        .sort_values("mean_loo", ascending=False)
    )
    ordered_aids = agg["annotator_id"].tolist()

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "<b>Leave-One-Out SD Impact (Mean ΔSD = SD_all − SD_without_a)</b>",
            "<b>Signed Timing Offset vs. Peers (t_a − mean(t_others))</b>",
        ),
        horizontal_spacing=0.09,
    )

    colors = ["#E74C3C" if v > 5.0 else ("#F59E0B" if v > 0 else "#3B82F6") for v in agg["mean_loo"]]
    fig.add_trace(
        go.Bar(
            x=agg["annotator_id"],
            y=agg["mean_loo"],
            error_y=dict(type="data", array=1.96 * agg["se_loo"], visible=True, thickness=1.2),
            marker_color=colors,
            marker_line=dict(width=0.7, color="#1f2937"),
            name="Mean LOO ΔSD (±95% CI)",
            showlegend=False,
            hovertemplate="<b>%{x}</b><br>Mean LOO ΔSD: %{y:+.2f} ms<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_hline(y=0.0, line_width=1.4, line_color="#374151", row=1, col=1)

    for idx, aid in enumerate(ordered_aids):
        sub_bias = ann_sd_df.loc[ann_sd_df["annotator_id"] == aid, "bias_vs_peers_ms"].dropna()
        fig.add_trace(
            go.Box(
                y=sub_bias,
                name=aid,
                marker_color=_PALETTE[idx % len(_PALETTE)],
                boxpoints=False,
                boxmean=True,
                showlegend=False,
            ),
            row=1,
            col=2,
        )
    fig.add_hline(y=0.0, line_width=1.4, line_dash="dash", line_color="#374151", row=1, col=2)

    fig.update_layout(
        title=dict(
            text=(
                "<b>Causal Annotator Diagnostics: Who Drives SD Increases?</b>"
                "<br><sup>Left: Positive ΔSD = removing this annotator reduces SD (outlier driver). "
                "Right: Positive/Negative = systematically late/early vs. peers.</sup>"
            ),
            x=0.5,
            font=dict(size=14),
        ),
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=_FONT,
        height=470,
        margin=dict(l=60, r=25, t=90, b=60),
    )
    _style_axes(fig)
    fig.update_xaxes(title_text="Annotator (sorted by LOO ΔSD)", row=1, col=1)
    fig.update_xaxes(title_text="Annotator (same order)", row=1, col=2)
    fig.update_yaxes(title_text="Mean LOO ΔSD (ms) ± 95% CI", row=1, col=1)
    fig.update_yaxes(title_text="Signed Offset vs. Peers (ms)", row=1, col=2)
    return fig

### 📋 Load Data & Per-Annotator SD Summary Table

In [3]:
# @title 📋 Load Annotator SD Records & Display Summary Table
ann_sd_df, consensus_all_df = load_annotator_sd_data(dataset_dir, min_annotators=3)
print(
    f"✅ Loaded {len(consensus_all_df):,} consensus marker instances (≥3 annotators) "
    f"across {consensus_all_df['set_id'].nunique()} sets and "
    f"{ann_sd_df['annotator_id'].nunique()} unique annotators ({len(ann_sd_df):,} annotator-marker records)."
)

summary_df = build_annotator_summary_table(ann_sd_df, consensus_all_df, threshold_ms=200.0)
display(summary_df)

✅ Loaded 2,952 consensus marker instances (≥3 annotators) across 54 sets and 13 unique annotators (8,856 annotator-marker records).


        Annotator  Sets  Instances  Median SD (ms)  Δ Med vs Overall (ms)  \
0   OVERALL (All)    54       2952            69.4                    0.0   
1              a5    12        720            88.2                   18.8   
2             a12    13        684            88.2                   18.8   
3              a6    12        696            88.2                   18.8   
4              a4    12        684            83.9                   14.5   
5             a10    13        696            77.0                    7.6   
6             a11    13        684            73.2                    3.8   
7              a7    11        636            69.4                    0.0   
8              a3    12        612            69.4                    0.0   
9              a8     5        276            69.4                    0.0   
10            a13    21       1176            68.0                   -1.4   
11             a1    13        660            66.7                   -2.7   

### 📊 1. Individual Annotator SD Distributions vs. Overall (Small Multiples)

In [4]:
# @title 📊 Plot Individual Annotator SD Distributions vs. Overall { display-mode: "form" }
bin_width_ms = 20.0 # @param {type:"number"}
sort_by = "median_sd" # @param ["median_sd", "mean_loo_delta_sd", "annotator_id"]
cols = 3 # @param {type:"integer"}
x_max_ms = 0 # @param {type:"number"}  # 0 = auto (99.5th pct)

fig_grid = plot_annotator_sd_small_multiples(
    ann_sd_df,
    consensus_all_df,
    bin_width_ms=bin_width_ms,
    sort_by=sort_by,
    x_max_ms=(None if x_max_ms <= 0 else x_max_ms),
    cols=cols,
)
fig_grid.show(config=EXPORT_CFG)

### 📈 2. Interactive Overlay: Normalized SD Curves & Empirical CDFs

In [5]:
# @title 📈 Overlay All Annotators vs. Overall (Density & ECDF)
fig_overlay = plot_annotator_sd_overlay(
    ann_sd_df,
    consensus_all_df,
    bin_width_ms=bin_width_ms,
    x_max_ms=(None if x_max_ms <= 0 else x_max_ms),
)
fig_overlay.show(config=EXPORT_CFG)

### 🎯 3. Causal Check: Leave-One-Out SD Impact ($\Delta\text{SD}_a$) & Signed Peer Bias

In [6]:
# @title 🎯 Leave-One-Out SD Impact & Signed Bias vs. Peers
fig_loo = plot_annotator_loo_impact(ann_sd_df)
fig_loo.show(config=EXPORT_CFG)